<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Meet the Larkfield data

**[Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/)** · Anomaly Detection: Find What Is Unusual · Module 1, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** load every Larkfield table of the course, check that your setup works, and score a first (silly) detector with the course's comparison table.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells (about 17 MB of data). No account. |
| **You should know** | The kinds of anomalies, from [Kinds of anomalies](https://learning.nextia-ai.com/courses/anomaly-detection/m01/kinds-of-anomalies/), and how to read a pandas table, from the machine learning course. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M01-L04-meet-the-larkfield-data/meet-the-larkfield-data-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `customers.csv`, `refunds_daily.csv`, `refunds_agents.csv`, `refunds.csv`, `ticket_api_metrics.csv.gz`, `ticket_api_logs.csv.gz`, `ticket_api_events.csv`, `orders.csv`, `tickets_daily.csv`, `calendar.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 20 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "customers.csv": "8410dc3b8ed4278f133040cd2f3cbcf5c241ed6492d7149f927c93815cd1d766",
    "refunds_daily.csv": "04c6384d1fbe8da908a739edc084eabc413581d75fce7cd84dcf7a90988996a6",
    "refunds_agents.csv": "1a883275f23988a40d99955e17f23ba132d98b7fb67912b0e197478cfff21782",
    "refunds.csv": "e35d78db18584188a2bae7742720796cc43d3801fcfc29c0c77b3e766ce28484",
    "ticket_api_metrics.csv.gz": "73b420194586f72b760fd672f89ee2526035385965668a3098cf1e34dfa123c4",
    "ticket_api_logs.csv.gz": "6ac2245f8473731f0d9e3f600aaa523f2b0ae3a701a81b288ad09010d1afdb0c",
    "ticket_api_events.csv": "3396099abee2b4326a301806fd7885b28f02e287ec1c455c6599bd280684d0c6",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Load every table

Larkfield is the online shop for home and garden products of all our courses. In this course, it wants to find unusual events before they cost money: stolen cards, refunds that do not fit, a failing service, a broken data feed. Every table is made up for the course, and every unusual event in it is one we planted on purpose.

Run the next cell. It reads the eleven tables and prints the number of rows of each one.

> **Check.** You should see `payments.csv.gz 343,074` at the top and `calendar.csv 819` at the end.

In [ ]:
files = ["payments.csv.gz", "customers.csv", "refunds_daily.csv", "refunds_agents.csv", "refunds.csv",
         "ticket_api_metrics.csv.gz", "ticket_api_logs.csv.gz", "ticket_api_events.csv", "orders.csv",
         "tickets_daily.csv", "calendar.csv"]
tables = {name: pd.read_csv(f"data/{name}") for name in files}
for name, table in tables.items():
    print(f"{name:<28}{len(table):>9,} rows   {', '.join(table.columns[:6])}{' ...' if len(table.columns) > 6 else ''}")

Four groups of tables, one for each kind of problem in the course:

| Tables | What they are | Used in |
|---|---|---|
| `payments`, `customers` | 343,074 card payments over six months (June to November 2025), and the customers | baselines, distance and isolation, deep methods, evaluation, the detection job |
| `refunds_daily`, `refunds_agents`, `refunds` | Larkfield's refunds over 21 months: per day, per agent and day, and one row per refund | control charts, drift tests, business checks |
| `ticket_api_metrics`, `ticket_api_logs`, `ticket_api_events` | 60 days of the help desk's `ticket-api` service: one row per minute, the log lines, the releases | anomalies in time, logs and embeddings |
| `orders`, `tickets_daily`, `calendar` | two years of daily orders and help-desk tickets per channel, with the holidays: the same numbers as in the forecasting course | anomalies in time |

## 2. The payment that did not fit

In the first lesson, you met a customer who buys a small basket every Saturday morning. Then, one night, eight payments come from Lisbon in a few minutes. Run the next cell to see this customer's last payments.

> **Check.** You should see Saturday payments of 11.15 to 17.98 dollars in Canada, then 8 payments in `PT` (Portugal) on 2025-10-03 between 02:14 and 02:21, on a new device.

In [ ]:
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
hook = payments[payments.customer_id == "C-01234"]
hook[["ts", "amount", "country", "device_id", "merchant_category"]].tail(12)

A person sees it at once: a new country, a new device, the middle of the night, eight payments in a few minutes. This is an anomaly **we planted**: the data has an answer key, and this payment is in it. The course's question is how a program can see the same thing among the other payments of the same six months, without a person looking at each one.

## 3. What we planted

The answer key of the course says which rows are unusual on purpose. You do not see it: the next sections show you how to score a detector **against** it, without reading it. These are the rules of what we planted (the full list is in the [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md)):

- **Payments:** stolen cards used for many small payments in a short time; card testing (one device tries many customers' cards with tiny payments); account takeovers (a few expensive payments).
- **Refunds:** the refund rate rises a little and stays there; one agent refunds far more than the others; one agent's refunds cluster just below the 100-dollar approval limit; some refunds are paid twice; the daily feed sends zeros for one day.
- **ticket-api:** a slow memory leak; an error spike after a release; a configuration change.

Some events look unusual but are **normal**: customers who travel, business customers' bulk orders, Black Friday, the new region `east`, and the Larkfield app launched on 2025-10-15. A good detector does not report them.

## 4. Daily orders, with Black Friday

Run the next cell. It adds up the orders of the three channels for each day and draws them, with Black Friday and the other holidays marked.

> **Check.** You should see a weekly up-and-down, two summers, and a red line at each Black Friday. On 2025-11-28 Larkfield had 1,224 orders.

In [ ]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"])
calendar = pd.read_csv("data/calendar.csv", parse_dates=["date"])
daily = orders.groupby("date").orders.sum()
holidays = calendar[(calendar.is_holiday == 1) & calendar.date.between(daily.index.min(), daily.index.max())]
fig, ax = plt.subplots()
ax.plot(daily.index, daily.to_numpy(), lw=0.8)
for d, name in zip(holidays.date, holidays.holiday):
    ax.axvline(d, color="red" if name == "Black Friday" else "grey", lw=1 if name == "Black Friday" else 0.5, alpha=0.7)
ax.set_xlabel("day")
ax.set_ylabel("orders (all channels)")
ax.set_title("Larkfield's daily orders; red: Black Friday, grey: other holidays")
plt.show()
print("Black Friday 2025:", int(daily.loc["2025-11-28"]), "orders")

Black Friday is far above every other day. It is unusual, but it is **normal** for Larkfield: the shop plans for it every year. Remember it: in Module 5, a good detector does not report Black Friday.

> **Predict.** The orders table has one day with no rows (the orders feed dropped it). Will the chart show a gap, a drop to zero, or nothing?

Run the next cell to find the day.

In [ ]:
all_days = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
missing = all_days.difference(daily.index)
print("Days with no orders rows:", [str(d.date()) for d in missing])

The line of the chart joins the day before and the day after, so you see **nothing**. A missing day is a data problem, and no detector that reads the values can see it: there is no value. A simple count of the rows per day finds it.

## 5. Meet the comparison table

Every module of the course runs its detectors on the same data and adds one row to one **comparison table**, with the same columns:

| Column | What it says |
|---|---|
| alerts per day | how many alerts the detector sends each day |
| precision at budget | of the alerts that Sofia's team reviews (50 a day, the **alert budget**), the share that are planted anomalies |
| recall | the share of all planted anomalies that the reviewed alerts find |
| cost per week ($) | the missed frauds (their amount plus a 15-dollar chargeback fee) and the false alerts (3 dollars each): **assumptions** of the course |
| explain one alert (min) | how long a reviewer needs to understand one alert (an assumption you state) |

`compare.py` computes the row. It also splits the payments the same way for everybody: detectors learn "normal" from the **reference months** (June to August) and are scored on the **evaluation months** (September to November).

Your first detector is silly on purpose: every day, alert on the 50 largest amounts.

> **Predict.** Of the 50 largest payments of a day, how many are planted frauds? Write down a number, then run the next cell.

In [ ]:
import compare
reference, evaluation = compare.split(payments)
table = compare.Table()
table.add(compare.score_payments(evaluation, evaluation.amount, "biggest amounts", "the 50 largest amounts of each day"))
table.show()

> **Check.** You should see a precision at budget of 0.003: about 0 of the 50 daily alerts are planted frauds, and the cost is 10,264 dollars a week.

Big amounts are not fraud: most frauds in the data are small payments. A large amount is often a business customer's normal order.

## 6. Change one thing: a random detector

Run the next cell. It adds a detector that picks 50 payments a day at random. Is the "biggest amounts" detector better than chance?

In [ ]:
rng = np.random.default_rng(0)
table.add(compare.score_payments(evaluation, rng.random(len(evaluation)), "random", "50 payments a day, chosen at random"))
table.show()

> **Check.** The random detector has a precision of about 0.019. The biggest amounts do worse than chance (0.003).

A baseline that a detector must beat is the first row of every comparison. In the next module, the simple baselines beat both rows by far.

## 7. A common mistake: scores in a different order

`compare.score_payments` needs one score per payment, **in the same order as the table** you give it. Run the next cell. It sorts the scores, but not the payments, before it scores them. The row is wrong, and nothing warns you.

In [ ]:
wrong = np.sort(evaluation.amount.to_numpy())          # sorted scores: no longer in the order of the payments
row = compare.score_payments(evaluation, wrong, "biggest amounts, wrong order", "scores sorted, payments not")
print(row["precision at budget"], "(the right order gave", table.rows[0]["precision at budget"], ")")

The fix: compute the score as a column of the same table (`evaluation.amount`, or `scores = model.score(...)` on the same rows), and never sort one without the other.

## 8. Check yourself

**Your turn 1.** How many payments are there on Black Friday, 2025-11-28? Write the number as `bf_payments`.

In [ ]:
bf_payments = int((payments.ts.dt.floor("D") == "2025-11-28").sum())

In [ ]:
expect('bf_payments', bf_payments, 5216)

**Your turn 2.** The refunds table also has days with no orders (the same feed). How many rows of `refunds_daily.csv` have an empty `orders`? Write the number as `missing_orders`.

In [ ]:
refunds_daily = pd.read_csv("data/refunds_daily.csv", parse_dates=["date"])
missing_orders = int(refunds_daily.orders.isna().sum())

In [ ]:
expect('missing_orders', missing_orders, 1)

**Your turn 3.** Score a third silly detector: every day, the 50 **smallest** amounts. Add its row to `table` and keep the row as `smallest`. Then compare it with the other two: why could small amounts find more planted frauds than large ones?

In [ ]:
smallest = compare.score_payments(evaluation, -evaluation.amount, "smallest amounts", "the 50 smallest amounts of each day")
table.add(smallest)
table.show()

In [ ]:
expect_hash('the precision at budget of the smallest amounts', smallest['precision at budget'] if isinstance(smallest, dict) else None, '6e87768dc78d643c')

Card testing uses tiny payments of 1 to 5 dollars, so the smallest amounts find some of them (0.192 precision in the solution). A rule about amounts alone still misses most frauds: the next module looks at each payment against the customer's own usual amount.

## Recap

- You loaded the eleven Larkfield tables: payments, refunds, the `ticket-api` service, and the daily orders and tickets.
- Some unusual events are planted problems; some are normal (Black Friday, travel, the app launch). A missing day is invisible in a chart.
- The comparison table scores every detector the same way: precision at the 50-alert budget, recall, cost per week. A silly detector is the first baseline.

If something broke, delete the folder `anomaly-course` and run the notebook again from the top.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Z-scores, and why the mean is fooled](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.